## 1. Data Preparation

### Objective
Prepare validated player-season and player-phase datasets for
evaluating KKR's batting and bowling contributions across IPL 2023
and 2024.

### Approach
1. Load the original delivery-level dataset.
2. Apply the established feature-engineering functions.
3. Generate player-season batting and bowling aggregates.
4. Generate player-phase batting and bowling aggregates.
5. Validate player identities, season coverage, and aggregation grain.

### Important considerations
- The dataset omits KKR's final two league matches in 2024.
- The represented 2024 sample contains 12 league matches and two
  knockout matches.
- Player comparisons must account for workload and opportunities.
- Bowling metrics retain the documented source-data inconsistencies.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

current_path = Path.cwd()

project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "src" / "config.py").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing src/config.py."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /Users/soham/MyDocs/Projects/Cricket Analysis/Hackathon - DPL 2026/CodeFiles/Hackathon_DPL2026


In [2]:
from src.data_loader import load_raw_data
from src.features.metric_definitions import add_metric_features
from src.features.player_level import (
    build_player_season_batting,
    build_player_season_bowling,
    build_player_season_phase_batting,
    build_player_season_phase_bowling,
)

df = load_raw_data()

df_features = add_metric_features(df)

print(f"Raw dataset: {df.shape}")
print(f"Feature dataset: {df_features.shape}")
print(f"Seasons available: {sorted(df_features['year'].dropna().unique())}")

Raw dataset: (34967, 61)
Feature dataset: (34967, 71)
Seasons available: [np.int64(2023), np.int64(2024)]


In [3]:
import inspect

functions_to_inspect = {
    "build_player_season_batting": build_player_season_batting,
    "build_player_season_bowling": build_player_season_bowling,
    "build_player_season_phase_batting": build_player_season_phase_batting,
    "build_player_season_phase_bowling": build_player_season_phase_bowling,
}

for name, func in functions_to_inspect.items():
    print(f"{name}{inspect.signature(func)}")

build_player_season_batting(df: 'pd.DataFrame') -> 'pd.DataFrame'
build_player_season_bowling(df: 'pd.DataFrame') -> 'pd.DataFrame'
build_player_season_phase_batting(df: 'pd.DataFrame') -> 'pd.DataFrame'
build_player_season_phase_bowling(df: 'pd.DataFrame') -> 'pd.DataFrame'


In [4]:
# Player-season aggregates
player_season_batting = build_player_season_batting(df_features)
player_season_bowling = build_player_season_bowling(df_features)

# Player-season-phase aggregates
player_season_phase_batting = build_player_season_phase_batting(df_features)
player_season_phase_bowling = build_player_season_phase_bowling(df_features)

# Display dataset dimensions
player_datasets = {
    "Player-season batting": player_season_batting,
    "Player-season bowling": player_season_bowling,
    "Player-phase batting": player_season_phase_batting,
    "Player-phase bowling": player_season_phase_bowling,
}

for name, dataset in player_datasets.items():
    print(f"{name}: {dataset.shape}")

Player-season batting: (347, 12)
Player-season bowling: (273, 18)
Player-phase batting: (733, 13)
Player-phase bowling: (672, 19)


In [5]:
# Inspect the schema of each dataset
for name, dataset in player_datasets.items():
    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")
    print("Columns:", dataset.columns.tolist())
    display(dataset.head(3))


Player-season batting
Columns: ['p_bat', 'year', 'batter_name', 'runs', 'balls_faced', 'dot_balls', 'boundary_balls', 'boundary_runs', 'strike_rate', 'dot_ball_pct', 'boundary_ball_pct', 'boundary_run_pct']


,p_bat,year,batter_name,runs,balls_faced,dot_balls,boundary_balls,boundary_runs,strike_rate,dot_ball_pct,boundary_ball_pct,boundary_run_pct
0,8917,2023,Moeen Ali,124,91,37,18,84,136.263736,40.659341,19.780220,67.741935
1,8917,2024,Moeen Ali,128,98,34,15,76,130.612245,34.693878,15.306122,59.375000
2,25913,2024,Mohammad Nabi,35,31,13,4,20,112.903226,41.935484,12.903226,57.142857



Player-season bowling
Columns: ['p_bowl', 'year', 'bowler_name', 'runs_conceded', 'legal_balls', 'dot_balls', 'wickets', 'boundary_balls', 'boundary_runs_conceded', 'wides_conceded', 'noball_extras_conceded', 'wide_noball_extras_conceded', 'overs_bowled', 'economy_rate', 'bowling_dot_pct', 'wickets_per_100_balls', 'boundary_balls_per_100_legal_balls', 'extras_per_20_overs']


,p_bowl,year,bowler_name,runs_conceded,legal_balls,dot_balls,wickets,boundary_balls,boundary_runs_conceded,wides_conceded,noball_extras_conceded,wide_noball_extras_conceded,overs_bowled,economy_rate,bowling_dot_pct,wickets_per_100_balls,boundary_balls_per_100_legal_balls,extras_per_20_overs
0,8917,2023,Moeen Ali,195,156,54,9,20,104,1,0,1,26.000000,7.500000,34.615385,5.769231,12.820513,0.769231
1,8917,2024,Moeen Ali,71,48,12,2,6,32,1,0,1,8.000000,8.875000,25.000000,4.166667,12.500000,2.500000
2,25913,2024,Mohammad Nabi,109,74,18,2,12,64,0,0,0,12.333333,8.837838,24.324324,2.702703,16.216216,0.000000



Player-phase batting
Columns: ['p_bat', 'year', 'match_phase', 'batter_name', 'runs', 'balls_faced', 'dot_balls', 'boundary_balls', 'boundary_runs', 'strike_rate', 'dot_ball_pct', 'boundary_ball_pct', 'boundary_run_pct']


,p_bat,year,match_phase,batter_name,runs,balls_faced,dot_balls,boundary_balls,boundary_runs,strike_rate,dot_ball_pct,boundary_ball_pct,boundary_run_pct
0,8917,2023,Death,Moeen Ali,39,24,7,5,26,162.500000,29.166667,20.833333,66.666667
1,8917,2023,Middle,Moeen Ali,62,50,19,8,36,124.000000,38.000000,16.000000,58.064516
2,8917,2023,Powerplay,Moeen Ali,23,17,11,5,22,135.294118,64.705882,29.411765,95.652174



Player-phase bowling
Columns: ['p_bowl', 'year', 'match_phase', 'bowler_name', 'runs_conceded', 'legal_balls', 'dot_balls', 'wickets', 'boundary_balls', 'boundary_runs_conceded', 'wides_conceded', 'noball_extras_conceded', 'wide_noball_extras_conceded', 'overs_bowled', 'economy_rate', 'bowling_dot_pct', 'wickets_per_100_balls', 'boundary_balls_per_100_legal_balls', 'extras_per_20_overs']


,p_bowl,year,match_phase,bowler_name,runs_conceded,legal_balls,dot_balls,wickets,boundary_balls,boundary_runs_conceded,wides_conceded,noball_extras_conceded,wide_noball_extras_conceded,overs_bowled,economy_rate,bowling_dot_pct,wickets_per_100_balls,boundary_balls_per_100_legal_balls,extras_per_20_overs
0,8917,2023,Death,Moeen Ali,3,6,3,1,0,0,0,0,0,1.0,3.000000,50.000000,16.666667,0.000000,0.000000
1,8917,2023,Middle,Moeen Ali,182,138,45,6,20,104,1,0,1,23.0,7.913043,32.608696,4.347826,14.492754,0.869565
2,8917,2023,Powerplay,Moeen Ali,10,12,6,2,0,0,0,0,0,2.0,5.000000,50.000000,16.666667,0.000000,0.000000


In [6]:
# Check the declared grain of each aggregation
grain_checks = {
    "Player-season batting": (
        player_season_batting,
        ["p_bat", "year"],
    ),
    "Player-season bowling": (
        player_season_bowling,
        ["p_bowl", "year"],
    ),
    "Player-phase batting": (
        player_season_phase_batting,
        ["p_bat", "year", "match_phase"],
    ),
    "Player-phase bowling": (
        player_season_phase_bowling,
        ["p_bowl", "year", "match_phase"],
    ),
}

for name, (dataset, grain_columns) in grain_checks.items():
    missing_columns = set(grain_columns) - set(dataset.columns)

    if missing_columns:
        print(f"{name}: MISSING GRAIN COLUMNS {missing_columns}")
        continue

    duplicate_count = dataset.duplicated(
        subset=grain_columns
    ).sum()

    print(
        f"{name}: "
        f"rows={len(dataset)}, "
        f"duplicate grain keys={duplicate_count}"
    )

Player-season batting: rows=347, duplicate grain keys=0
Player-season bowling: rows=273, duplicate grain keys=0
Player-phase batting: rows=733, duplicate grain keys=0
Player-phase bowling: rows=672, duplicate grain keys=0


In [8]:
# Identify KKR batter and bowler IDs from the feature-engineered data
kkr_batter_ids = (
    df_features.loc[
        df_features["team_bat"].eq("Kolkata Knight Riders"),
        ["p_bat", "bat"]
    ]
    .drop_duplicates()
)

kkr_bowler_ids = (
    df_features.loc[
        df_features["team_bowl"].eq("Kolkata Knight Riders"),
        ["p_bowl", "bowl"]
    ]
    .drop_duplicates()
)

# Filter the player-season aggregates
kkr_batting = player_season_batting.loc[
    player_season_batting["p_bat"].isin(kkr_batter_ids["p_bat"])
].copy()

kkr_bowling = player_season_bowling.loc[
    player_season_bowling["p_bowl"].isin(kkr_bowler_ids["p_bowl"])
].copy()

# Filter the player-phase aggregates
kkr_batting_phase = player_season_phase_batting.loc[
    player_season_phase_batting["p_bat"].isin(kkr_batter_ids["p_bat"])
].copy()

kkr_bowling_phase = player_season_phase_bowling.loc[
    player_season_phase_bowling["p_bowl"].isin(kkr_bowler_ids["p_bowl"])
].copy()

# Display coverage
print("KKR batting player-season records:", len(kkr_batting))
print("KKR bowling player-season records:", len(kkr_bowling))
print("KKR batting phase records:", len(kkr_batting_phase))
print("KKR bowling phase records:", len(kkr_bowling_phase))

print("\nBatting season coverage:")
display(
    kkr_batting.groupby("year")
    .agg(
        players=("p_bat", "nunique"),
        player_season_records=("p_bat", "size"),
    )
    .reset_index()
)

print("\nBowling season coverage:")
display(
    kkr_bowling.groupby("year")
    .agg(
        players=("p_bowl", "nunique"),
        player_season_records=("p_bowl", "size"),
    )
    .reset_index()
)

KKR batting player-season records: 35
KKR bowling player-season records: 28
KKR batting phase records: 76
KKR bowling phase records: 73

Batting season coverage:


,year,players,player_season_records
0,2023,19,19
1,2024,16,16



Bowling season coverage:


,year,players,player_season_records
0,2023,14,14
1,2024,14,14


In [10]:
# Identify KKR batter-season combinations
kkr_batter_seasons = (
    df_features.loc[
        df_features["team_bat"].eq("Kolkata Knight Riders"),
        ["p_bat", "year"]
    ]
    .drop_duplicates()
)

# Identify KKR bowler-season combinations
kkr_bowler_seasons = (
    df_features.loc[
        df_features["team_bowl"].eq("Kolkata Knight Riders"),
        ["p_bowl", "year"]
    ]
    .drop_duplicates()
)

# Player-season aggregates: one row per player-season
kkr_batting = player_season_batting.merge(
    kkr_batter_seasons,
    on=["p_bat", "year"],
    how="inner",
    validate="one_to_one",
)

kkr_bowling = player_season_bowling.merge(
    kkr_bowler_seasons,
    on=["p_bowl", "year"],
    how="inner",
    validate="one_to_one",
)

# Player-phase aggregates: multiple phases per player-season
kkr_batting_phase = player_season_phase_batting.merge(
    kkr_batter_seasons,
    on=["p_bat", "year"],
    how="inner",
    validate="many_to_one",
)

kkr_bowling_phase = player_season_phase_bowling.merge(
    kkr_bowler_seasons,
    on=["p_bowl", "year"],
    how="inner",
    validate="many_to_one",
)

# Report the resulting dataset sizes
print("KKR batting player-season records:", len(kkr_batting))
print("KKR bowling player-season records:", len(kkr_bowling))
print("KKR batting phase records:", len(kkr_batting_phase))
print("KKR bowling phase records:", len(kkr_bowling_phase))

KKR batting player-season records: 31
KKR bowling player-season records: 25
KKR batting phase records: 67
KKR bowling phase records: 64


In [11]:
# Select KKR Powerplay batting records
kkr_powerplay_batting = kkr_batting_phase.loc[
    kkr_batting_phase["match_phase"].eq("Powerplay")
].copy()

powerplay_batting_comparison = (
    kkr_powerplay_batting[
        [
            "year",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
            "boundary_run_pct",
        ]
    ]
    .sort_values(
        ["year", "runs"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

display(powerplay_batting_comparison)

,year,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct,boundary_run_pct
0,2023,Jason Roy,177,121,146.280992,48.760331,27.272727,81.355932
1,2023,Rahmanullah Gurbaz,145,112,129.464286,53.571429,19.642857,74.482759
2,2023,Venkatesh Iyer,142,109,130.275229,50.458716,21.100917,74.647887
3,2023,Nitish Rana,94,71,132.394366,49.295775,21.126761,72.340426
4,2023,Narayan Jagadeesan,63,58,108.620690,55.172414,18.965517,73.015873
5,2023,Rinku Singh,18,11,163.636364,36.363636,27.272727,77.777778
6,2023,Mandeep Singh,4,8,50.000000,50.000000,0.000000,0.000000
7,2023,Litton Das,4,4,100.000000,75.000000,25.000000,100.000000
8,2023,Anukul Roy,4,5,80.000000,80.000000,20.000000,100.000000
9,2023,Sunil Narine,0,4,0.000000,100.000000,0.000000,NaN


In [12]:
powerplay_batting_screen = (
    kkr_powerplay_batting.loc[
        kkr_powerplay_batting["balls_faced"].ge(30)
    ]
    .sort_values(
        ["year", "strike_rate"],
        ascending=[True, False],
    )
    [
        [
            "year",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
        ]
    ]
    .reset_index(drop=True)
)

display(powerplay_batting_screen)

,year,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
0,2023,Jason Roy,177,121,146.280992,48.760331,27.272727
1,2023,Nitish Rana,94,71,132.394366,49.295775,21.126761
2,2023,Venkatesh Iyer,142,109,130.275229,50.458716,21.100917
3,2023,Rahmanullah Gurbaz,145,112,129.464286,53.571429,19.642857
4,2023,Narayan Jagadeesan,63,58,108.620690,55.172414,18.965517
5,2024,Venkatesh Iyer,111,50,222.000000,34.000000,44.000000
6,2024,Phil Salt,296,160,185.000000,40.625000,33.125000
7,2024,Sunil Narine,281,165,170.303030,46.060606,30.303030
8,2024,Angkrish Raghuvanshi,78,49,159.183673,42.857143,26.530612
9,2024,Rahmanullah Gurbaz,44,36,122.222222,58.333333,22.222222


In [13]:
# Leading KKR Powerplay contributors by season
powerplay_leaders = (
    kkr_powerplay_batting
    .sort_values(
        ["year", "runs"],
        ascending=[True, False],
    )
    .groupby("year", group_keys=False)
    .head(5)
    [
        [
            "year",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
        ]
    ]
    .reset_index(drop=True)
)

display(
    powerplay_leaders.round(2)
)

,year,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
0,2023,Jason Roy,177,121,146.28,48.76,27.27
1,2023,Rahmanullah Gurbaz,145,112,129.46,53.57,19.64
2,2023,Venkatesh Iyer,142,109,130.28,50.46,21.10
3,2023,Nitish Rana,94,71,132.39,49.30,21.13
4,2023,Narayan Jagadeesan,63,58,108.62,55.17,18.97
5,2024,Phil Salt,296,160,185.00,40.62,33.12
6,2024,Sunil Narine,281,165,170.30,46.06,30.30
7,2024,Venkatesh Iyer,111,50,222.00,34.00,44.00
8,2024,Angkrish Raghuvanshi,78,49,159.18,42.86,26.53
9,2024,Rahmanullah Gurbaz,44,36,122.22,58.33,22.22


In [15]:
# Select KKR middle-over batting records
kkr_middle_batting = kkr_batting_phase.loc[
    kkr_batting_phase["match_phase"].eq("Middle")
].copy()

middle_batting_comparison = (
    kkr_middle_batting[
        [
            "year",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
            "boundary_run_pct",
        ]
    ]
    .sort_values(
        ["year", "runs"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

display(middle_batting_comparison.round(2))

,year,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct,boundary_run_pct
0,2023,Nitish Rana,269,193,139.38,33.68,19.17,63.94
1,2023,Venkatesh Iyer,234,146,160.27,18.49,17.81,55.56
2,2023,Rinku Singh,176,156,112.82,31.41,10.26,43.18
3,2023,Jason Roy,108,67,161.19,25.37,22.39,66.67
4,2023,Andre Russell,93,74,125.68,41.89,14.86,60.22
5,2023,Rahmanullah Gurbaz,82,57,143.86,35.09,17.54,60.98
6,2023,Shardul Thakur,61,27,225.93,11.11,33.33,72.13
7,2023,Narayan Jagadeesan,26,23,113.04,47.83,17.39,61.54
8,2023,Mandeep Singh,10,8,125.00,50.00,12.50,60.00
9,2023,Sunil Narine,5,7,71.43,71.43,14.29,80.00


In [16]:
middle_batting_screen = (
    kkr_middle_batting.loc[
        kkr_middle_batting["balls_faced"].ge(50)
    ]
    .sort_values(
        ["year", "runs"],
        ascending=[True, False],
    )
    [
        [
            "year",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
        ]
    ]
    .reset_index(drop=True)
)

display(middle_batting_screen.round(2))

,year,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
0,2023,Nitish Rana,269,193,139.38,33.68,19.17
1,2023,Venkatesh Iyer,234,146,160.27,18.49,17.81
2,2023,Rinku Singh,176,156,112.82,31.41,10.26
3,2023,Jason Roy,108,67,161.19,25.37,22.39
4,2023,Andre Russell,93,74,125.68,41.89,14.86
5,2023,Rahmanullah Gurbaz,82,57,143.86,35.09,17.54
6,2024,Shreyas Iyer,220,157,140.13,28.66,18.47
7,2024,Venkatesh Iyer,199,140,142.14,34.29,17.14
8,2024,Sunil Narine,177,93,190.32,31.18,29.03
9,2024,Phil Salt,134,76,176.32,27.63,26.32


In [17]:
# Select KKR death-over batting records
kkr_death_batting = kkr_batting_phase.loc[
    kkr_batting_phase["match_phase"].eq("Death")
].copy()

death_batting_comparison = (
    kkr_death_batting[
        [
            "year",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
            "boundary_run_pct",
        ]
    ]
    .sort_values(
        ["year", "runs"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

display(death_batting_comparison.round(2))

,year,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct,boundary_run_pct
0,2023,Rinku Singh,280,150,186.67,32.00,27.33,74.29
1,2023,Andre Russell,134,82,163.41,43.90,24.39,77.61
2,2023,Shardul Thakur,52,39,133.33,35.90,20.51,61.54
3,2023,Nitish Rana,50,29,172.41,37.93,24.14,72.00
4,2023,Venkatesh Iyer,28,22,127.27,36.36,18.18,64.29
5,2023,David Wiese,21,11,190.91,45.45,27.27,85.71
6,2023,Anukul Roy,19,13,146.15,15.38,15.38,42.11
7,2023,Umesh Yadav,18,17,105.88,35.29,5.88,22.22
8,2023,Sunil Narine,16,14,114.29,42.86,14.29,62.50
9,2023,Vaibhav Arora,3,2,150.00,0.00,0.00,0.00


In [18]:
death_batting_screen = (
    kkr_death_batting.loc[
        kkr_death_batting["balls_faced"].ge(30)
    ]
    .sort_values(
        ["year", "runs"],
        ascending=[True, False],
    )
    [
        [
            "year",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
        ]
    ]
    .reset_index(drop=True)
)

display(death_batting_screen.round(2))

,year,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
0,2023,Rinku Singh,280,150,186.67,32.00,27.33
1,2023,Andre Russell,134,82,163.41,43.90,24.39
2,2023,Shardul Thakur,52,39,133.33,35.90,20.51
3,2024,Andre Russell,135,64,210.94,29.69,34.38
4,2024,Shreyas Iyer,113,56,201.79,19.64,28.57
5,2024,Rinku Singh,86,48,179.17,27.08,25.00
6,2024,Venkatesh Iyer,60,43,139.53,44.19,18.60


### 2.4 Consolidated batting scouting

This section brings together the Powerplay, middle-over and death-over
analyses to identify role-specific strengths and potential areas for
improvement.

The assessment distinguishes production from efficiency and accounts
for differences in opportunities between players and seasons.

In [19]:
# Leading run-scorers by season and batting phase
batting_phase_leaders = (
    kkr_batting_phase
    .sort_values(
        ["year", "match_phase", "runs"],
        ascending=[True, True, False],
    )
    .groupby(
        ["year", "match_phase"],
        group_keys=False,
    )
    .head(5)
    [
        [
            "year",
            "match_phase",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
        ]
    ]
    .sort_values(
        ["year", "match_phase", "runs"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

display(batting_phase_leaders.round(2))

,year,match_phase,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
0,2023,Death,Rinku Singh,280,150,186.67,32.00,27.33
1,2023,Death,Andre Russell,134,82,163.41,43.90,24.39
2,2023,Death,Shardul Thakur,52,39,133.33,35.90,20.51
3,2023,Death,Nitish Rana,50,29,172.41,37.93,24.14
4,2023,Death,Venkatesh Iyer,28,22,127.27,36.36,18.18
5,2023,Middle,Nitish Rana,269,193,139.38,33.68,19.17
6,2023,Middle,Venkatesh Iyer,234,146,160.27,18.49,17.81
7,2023,Middle,Rinku Singh,176,156,112.82,31.41,10.26
8,2023,Middle,Jason Roy,108,67,161.19,25.37,22.39
9,2023,Middle,Andre Russell,93,74,125.68,41.89,14.86


## 3. Bowling Scouting

The bowling analysis evaluates KKR bowlers across three phases:
Powerplay (overs 1–6), Middle (overs 7–15), and Death (overs 16–20).

The assessment considers:
- Wickets taken and wicket-taking frequency
- Economy rate, expressed as runs per six legal balls
- Dot-ball percentage
- Boundary frequency and boundary runs conceded per over
- Wides, no-ball extras, workload, and matches represented

Wickets and economy are interpreted together. A bowler may take wickets
while conceding heavily, or build pressure through dot balls without
taking frequent wickets.

The analysis is descriptive and does not establish causality.

In [20]:
# Display KKR bowling metrics by player, season, and phase
bowling_phase_comparison = (
    kkr_bowling_phase[
        [
            "year",
            "match_phase",
            "bowler_name",
            "overs_bowled",
            "runs_conceded",
            "wickets",
            "economy_rate",
            "bowling_dot_pct",
            "boundary_balls_per_100_legal_balls",
            "boundary_runs_conceded",
            "wides_conceded",
            "noball_extras_conceded",
        ]
    ]
    .sort_values(
        ["year", "match_phase", "wickets"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

display(bowling_phase_comparison.round(2))

,year,match_phase,bowler_name,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,boundary_runs_conceded,wides_conceded,noball_extras_conceded
0,2023,Death,Varun Chakravarthy,12.67,106,4,8.37,31.58,14.47,62,3,0
1,2023,Death,Sunil Narine,8.00,69,3,8.62,25.00,14.58,32,2,0
2,2023,Death,Shardul Thakur,7.83,88,3,11.23,23.40,25.53,60,3,0
3,2023,Death,Vaibhav Arora,6.00,67,3,11.17,22.22,22.22,32,8,2
4,2023,Death,Andre Russell,5.17,60,2,11.61,25.81,22.58,38,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
59,2024,Powerplay,Harshit Rana,14.00,139,4,9.93,44.05,26.19,106,4,0
60,2024,Powerplay,Varun Chakravarthy,4.00,34,2,8.50,37.50,20.83,24,0,0
61,2024,Powerplay,Sunil Narine,7.00,64,1,9.14,42.86,23.81,48,1,0
62,2024,Powerplay,Dushmantha Chameera,2.00,30,0,15.00,16.67,41.67,24,1,0


In [21]:
# Screen bowlers with at least 60 legal balls in a phase
bowling_phase_screen = (
    kkr_bowling_phase.loc[
        kkr_bowling_phase["legal_balls"].ge(60)
    ]
    .copy()
)

# Calculate wickets per 20 overs for easier cricket interpretation
bowling_phase_screen["wickets_per_20_overs"] = (
    bowling_phase_screen["wickets"]
    / bowling_phase_screen["overs_bowled"]
    * 20
)

bowling_phase_screen = (
    bowling_phase_screen[
        [
            "year",
            "match_phase",
            "bowler_name",
            "overs_bowled",
            "runs_conceded",
            "wickets",
            "wickets_per_20_overs",
            "economy_rate",
            "bowling_dot_pct",
            "boundary_balls_per_100_legal_balls",
            "boundary_runs_conceded",
            "wides_conceded",
            "noball_extras_conceded",
        ]
    ]
    .sort_values(
        ["year", "match_phase", "wickets"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

display(bowling_phase_screen.round(2))

,year,match_phase,bowler_name,overs_bowled,runs_conceded,wickets,wickets_per_20_overs,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,boundary_runs_conceded,wides_conceded,noball_extras_conceded
0,2023,Death,Varun Chakravarthy,12.67,106,4,6.32,8.37,31.58,14.47,62,3,0
1,2023,Middle,Varun Chakravarthy,24.00,190,11,9.17,7.92,36.11,15.97,114,2,0
2,2023,Middle,Sunil Narine,33.00,245,6,3.64,7.42,26.26,10.61,104,3,0
3,2023,Middle,Suyash Sharma,30.00,231,6,4.00,7.70,35.56,15.56,134,3,0
4,2023,Powerplay,Varun Chakravarthy,16.00,133,5,6.25,8.31,40.62,21.88,88,7,0
5,2023,Powerplay,Harshit Rana,14.00,111,4,5.71,7.93,47.62,22.62,84,4,1
6,2023,Powerplay,Umesh Yadav,14.00,123,0,0.00,8.79,42.86,22.62,92,1,0
7,2024,Death,Harshit Rana,14.17,129,8,11.29,9.11,32.94,18.82,80,5,0
8,2024,Death,Mitchell Starc,10.83,134,6,11.08,12.37,38.46,27.69,98,13,0
9,2024,Middle,Varun Chakravarthy,38.00,292,17,8.95,7.68,39.91,15.35,172,7,2


In [22]:
# Leading wicket-takers by season and bowling phase
bowling_phase_leaders = (
    kkr_bowling_phase
    .sort_values(
        ["year", "match_phase", "wickets", "overs_bowled"],
        ascending=[True, True, False, False],
    )
    .groupby(
        ["year", "match_phase"],
        group_keys=False,
    )
    .head(5)
    [
        [
            "year",
            "match_phase",
            "bowler_name",
            "overs_bowled",
            "runs_conceded",
            "wickets",
            "economy_rate",
            "bowling_dot_pct",
            "boundary_balls_per_100_legal_balls",
            "wides_conceded",
            "noball_extras_conceded",
        ]
    ]
    .reset_index(drop=True)
)

display(bowling_phase_leaders.round(2))

,year,match_phase,bowler_name,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
0,2023,Death,Varun Chakravarthy,12.67,106,4,8.37,31.58,14.47,3,0
1,2023,Death,Sunil Narine,8.00,69,3,8.62,25.00,14.58,2,0
2,2023,Death,Shardul Thakur,7.83,88,3,11.23,23.40,25.53,3,0
3,2023,Death,Vaibhav Arora,6.00,67,3,11.17,22.22,22.22,8,2
4,2023,Death,Andre Russell,5.17,60,2,11.61,25.81,22.58,4,0
5,2023,Middle,Varun Chakravarthy,24.00,190,11,7.92,36.11,15.97,2,0
6,2023,Middle,Sunil Narine,33.00,245,6,7.42,26.26,10.61,3,0
7,2023,Middle,Suyash Sharma,30.00,231,6,7.70,35.56,15.56,3,0
8,2023,Middle,Shardul Thakur,9.17,82,3,8.95,30.91,20.00,0,2
9,2023,Middle,Anukul Roy,7.00,59,2,8.43,42.86,19.05,1,0


## League-Wide Scouting Candidate Pool

In [35]:
# Build player-to-team mappings from the delivery-level dataset.
# Player aggregations do not contain team names, so we recover
# team affiliation from the original delivery records.

batter_team_season = (
    df_features[
        ["p_bat", "year", "team_bat"]
    ]
    .drop_duplicates()
    .rename(columns={
        "p_bat": "player_id",
        "team_bat": "team",
    })
)

bowler_team_season = (
    df_features[
        ["p_bowl", "year", "team_bowl"]
    ]
    .drop_duplicates()
    .rename(columns={
        "p_bowl": "player_id",
        "team_bowl": "team",
    })
)

# Check whether a player is associated with multiple teams
# within the same season.

batter_team_conflicts = (
    batter_team_season
    .groupby(["player_id", "year"])["team"]
    .nunique()
    .loc[lambda x: x > 1]
)

bowler_team_conflicts = (
    bowler_team_season
    .groupby(["player_id", "year"])["team"]
    .nunique()
    .loc[lambda x: x > 1]
)

print("Batter player-season team conflicts:", len(batter_team_conflicts))
print("Bowler player-season team conflicts:", len(bowler_team_conflicts))

display(
    batter_team_season
    .groupby(["year", "team"])["player_id"]
    .nunique()
    .rename("distinct_batters")
    .reset_index()
    .sort_values(["year", "team"])
)

display(
    bowler_team_season
    .groupby(["year", "team"])["player_id"]
    .nunique()
    .rename("distinct_bowlers")
    .reset_index()
    .sort_values(["year", "team"])
)

Batter player-season team conflicts: 0
Bowler player-season team conflicts: 0


,year,team,distinct_batters
0,2023,Chennai Super Kings,12
1,2023,Delhi Capitals,22
2,2023,Gujarat Titans,15
3,2023,Kolkata Knight Riders,17
4,2023,Lucknow Super Giants,22
5,2023,Mumbai Indians,18
6,2023,Punjab Kings,18
7,2023,Rajasthan Royals,15
8,2023,Royal Challengers Bengaluru,19
9,2023,Sunrisers Hyderabad,19


,year,team,distinct_bowlers
0,2023,Chennai Super Kings,12
1,2023,Delhi Capitals,12
2,2023,Gujarat Titans,11
3,2023,Kolkata Knight Riders,14
4,2023,Lucknow Super Giants,16
5,2023,Mumbai Indians,15
6,2023,Punjab Kings,12
7,2023,Rajasthan Royals,13
8,2023,Royal Challengers Bengaluru,15
9,2023,Sunrisers Hyderabad,16


In [36]:
print("Batting phase table:", kkr_batting_phase.shape)
print("Bowling phase table:", kkr_bowling_phase.shape)

print("\nBatting phase columns:")
print(kkr_batting_phase.columns.tolist())

print("\nBowling phase columns:")
print(kkr_bowling_phase.columns.tolist())

print("\nLeague-wide batting phase table:", player_season_phase_batting.shape)
print("League-wide bowling phase table:", player_season_phase_bowling.shape)

Batting phase table: (67, 13)
Bowling phase table: (64, 19)

Batting phase columns:
['p_bat', 'year', 'match_phase', 'batter_name', 'runs', 'balls_faced', 'dot_balls', 'boundary_balls', 'boundary_runs', 'strike_rate', 'dot_ball_pct', 'boundary_ball_pct', 'boundary_run_pct']

Bowling phase columns:
['p_bowl', 'year', 'match_phase', 'bowler_name', 'runs_conceded', 'legal_balls', 'dot_balls', 'wickets', 'boundary_balls', 'boundary_runs_conceded', 'wides_conceded', 'noball_extras_conceded', 'wide_noball_extras_conceded', 'overs_bowled', 'economy_rate', 'bowling_dot_pct', 'wickets_per_100_balls', 'boundary_balls_per_100_legal_balls', 'extras_per_20_overs']

League-wide batting phase table: (733, 13)
League-wide bowling phase table: (672, 19)


In [37]:
# Attach team affiliation to league-wide player-phase batting statistics.

batting_candidates = player_season_phase_batting.merge(
    batter_team_season,
    left_on=["p_bat", "year"],
    right_on=["player_id", "year"],
    how="left",
    validate="many_to_one",
)

# Attach team affiliation to league-wide player-phase bowling statistics.

bowling_candidates = player_season_phase_bowling.merge(
    bowler_team_season,
    left_on=["p_bowl", "year"],
    right_on=["player_id", "year"],
    how="left",
    validate="many_to_one",
)

# Check team mapping before filtering.

print(
    "Batting records without team:",
    int(batting_candidates["team"].isna().sum()),
)

print(
    "Bowling records without team:",
    int(bowling_candidates["team"].isna().sum()),
)

# Exclude KKR players from the external recruitment pool.

batting_candidates = batting_candidates.loc[
    batting_candidates["team"].notna()
    & batting_candidates["team"].ne("Kolkata Knight Riders")
].copy()

bowling_candidates = bowling_candidates.loc[
    bowling_candidates["team"].notna()
    & bowling_candidates["team"].ne("Kolkata Knight Riders")
].copy()

print("\nNon-KKR batting phase records:", len(batting_candidates))
print("Non-KKR bowling phase records:", len(bowling_candidates))

display(
    batting_candidates
    .groupby(["year", "match_phase"])
    .size()
    .rename("candidate_records")
    .reset_index()
)

display(
    bowling_candidates
    .groupby(["year", "match_phase"])
    .size()
    .rename("candidate_records")
    .reset_index()
)

Batting records without team: 0
Bowling records without team: 0

Non-KKR batting phase records: 666
Non-KKR bowling phase records: 608


,year,match_phase,candidate_records
0,2023,Death,145
1,2023,Middle,121
2,2023,Powerplay,76
3,2024,Death,131
4,2024,Middle,118
5,2024,Powerplay,75


,year,match_phase,candidate_records
0,2023,Death,94
1,2023,Middle,118
2,2023,Powerplay,96
3,2024,Death,88
4,2024,Middle,118
5,2024,Powerplay,94


In [38]:
# Minimum opportunity by batting phase.

phase_ball_thresholds = {
    "Powerplay": MIN_PP_BALLS,
    "Middle": MIN_MIDDLE_BALLS,
    "Death": MIN_DEATH_BALLS,
}

batting_candidate_shortlists = {}

for year in [2023, 2024]:
    for phase, min_balls in phase_ball_thresholds.items():

        candidates = batting_candidates.loc[
            batting_candidates["year"].eq(year)
            & batting_candidates["match_phase"].eq(phase)
            & batting_candidates["balls_faced"].ge(min_balls)
        ].copy()

        candidates = candidates.sort_values(
            ["strike_rate", "runs", "balls_faced"],
            ascending=[False, False, False],
        )

        shortlist = candidates[
            [
                "year",
                "match_phase",
                "team",
                "batter_name",
                "runs",
                "balls_faced",
                "strike_rate",
                "dot_ball_pct",
                "boundary_ball_pct",
            ]
        ].head(10)

        batting_candidate_shortlists[(year, phase)] = shortlist

        print(
            f"\n{year} | {phase} | "
            f"Eligible candidates: {len(candidates)}"
        )

        display(shortlist.round(2))


2023 | Powerplay | Eligible candidates: 28


,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
99,2023,Powerplay,Chennai Super Kings,Ajinkya Rahane,125,60,208.33,23.33,31.67
620,2023,Powerplay,Rajasthan Royals,Yashasvi Jaiswal,361,206,175.24,46.60,34.47
551,2023,Powerplay,Mumbai Indians,Cameron Green,167,96,173.96,31.25,30.21
46,2023,Powerplay,Royal Challengers Bengaluru,Faf du Plessis,359,213,168.54,34.74,27.23
587,2023,Powerplay,Punjab Kings,Atharva Taide,102,65,156.92,46.15,26.15
380,2023,Powerplay,Delhi Capitals,Phil Salt,108,71,152.11,39.44,23.94
534,2023,Powerplay,Gujarat Titans,Shubman Gill,356,236,150.85,36.86,24.58
520,2023,Powerplay,Chennai Super Kings,Ruturaj Gaikwad,345,235,146.81,37.02,20.85
52,2023,Powerplay,Delhi Capitals,David Warner,299,205,145.85,48.78,27.80
416,2023,Powerplay,Mumbai Indians,Ishan Kishan,333,230,144.78,43.91,23.91



2023 | Middle | Eligible candidates: 43


,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
260,2023,Middle,Mumbai Indians,Suryakumar Yadav,450,246,182.93,25.20,28.05
160,2023,Middle,Royal Challengers Bengaluru,Glenn Maxwell,249,141,176.60,29.08,23.40
345,2023,Middle,Lucknow Super Giants,Nicholas Pooran,176,104,169.23,37.50,26.92
249,2023,Middle,Sunrisers Hyderabad,Heinrich Klaasen,249,148,168.24,18.92,19.59
533,2023,Middle,Gujarat Titans,Shubman Gill,429,272,157.72,20.59,17.28
642,2023,Middle,Punjab Kings,Prabhsimran Singh,105,68,154.41,30.88,17.65
421,2023,Middle,Punjab Kings,Jitesh Sharma,187,122,153.28,35.25,21.31
403,2023,Middle,Chennai Super Kings,Shivam Dube,288,189,152.38,30.69,16.93
243,2023,Middle,Rajasthan Royals,Sanju Samson,242,159,152.20,30.82,18.24
519,2023,Middle,Chennai Super Kings,Ruturaj Gaikwad,230,153,150.33,28.76,16.99



2023 | Death | Eligible candidates: 47


...

,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
649,2024,Powerplay,Delhi Capitals,Jake Fraser-McGurk,266,106,250.94,36.79,49.06
309,2024,Powerplay,Sunrisers Hyderabad,Travis Head,402,193,208.29,33.16,38.34
545,2024,Powerplay,Sunrisers Hyderabad,Abhishek Sharma,343,169,202.96,34.91,33.14
419,2024,Powerplay,Mumbai Indians,Ishan Kishan,239,148,161.49,43.92,28.38
80,2024,Powerplay,Royal Challengers Bengaluru,Virat Kohli,373,231,161.47,36.36,24.68
645,2024,Powerplay,Punjab Kings,Prabhsimran Singh,236,148,159.46,45.95,27.70
49,2024,Powerplay,Royal Challengers Bengaluru,Faf du Plessis,309,194,159.28,44.85,28.87
531,2024,Powerplay,Delhi Capitals,Prithvi Shaw,140,88,159.09,39.77,30.68
717,2024,Powerplay,Delhi Capitals,Abishek Porel,140,89,157.30,44.94,26.97
269,2024,Powerplay,Sunrisers Hyderabad,Rahul Tripathi,104,67,155.22,41.79,26.87



2024 | Middle | Eligible candidates: 48


,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
544,2024,Middle,Sunrisers Hyderabad,Abhishek Sharma,141,68,207.35,30.88,32.35
461,2024,Middle,Royal Challengers Bengaluru,Rajat Patidar,312,172,181.40,28.49,22.67
145,2024,Middle,Punjab Kings,Rilee Rossouw,114,63,180.95,28.57,25.40
348,2024,Middle,Lucknow Super Giants,Nicholas Pooran,247,143,172.73,23.78,22.38
48,2024,Middle,Royal Challengers Bengaluru,Faf du Plessis,126,73,172.60,10.96,16.44
484,2024,Middle,Royal Challengers Bengaluru,Will Jacks,129,77,167.53,31.17,23.38
263,2024,Middle,Mumbai Indians,Suryakumar Yadav,222,134,165.67,33.58,23.88
119,2024,Middle,Punjab Kings,Jonny Bairstow,118,72,163.89,37.50,23.61
622,2024,Middle,Rajasthan Royals,Yashasvi Jaiswal,136,84,161.90,23.81,19.05
308,2024,Middle,Sunrisers Hyderabad,Travis Head,165,103,160.19,27.18,21.36



2024 | Death | Eligible candidates: 41


,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
333,2024,Death,Delhi Capitals,Tristan Stubbs,252,96,262.50,11.46,39.58
497,2024,Death,Delhi Capitals,Rishabh Pant,133,59,225.42,23.73,33.90
191,2024,Death,Punjab Kings,Shashank Singh,135,61,221.31,19.67,32.79
16,2024,Death,Chennai Super Kings,MS Dhoni,161,73,220.55,30.14,36.99
25,2024,Death,Royal Challengers Bengaluru,Dinesh Karthik,249,116,214.66,27.59,33.62
251,2024,Death,Sunrisers Hyderabad,Heinrich Klaasen,195,91,214.29,23.08,28.57
134,2024,Death,Rajasthan Royals,Jos Buttler,72,35,205.71,25.71,25.71
150,2024,Death,Gujarat Titans,David Miller,110,54,203.70,22.22,29.63
552,2024,Death,Royal Challengers Bengaluru,Cameron Green,117,58,201.72,13.79,27.59
669,2024,Death,Sunrisers Hyderabad,Abdul Samad,121,60,201.67,23.33,30.00


In [39]:
bowling_candidate_shortlists = {}

for year in [2023, 2024]:
    for phase in ["Powerplay", "Middle", "Death"]:

        candidates = bowling_candidates.loc[
            bowling_candidates["year"].eq(year)
            & bowling_candidates["match_phase"].eq(phase)
            & (
                bowling_candidates["legal_balls"]
                >= MIN_BOWLING_BALLS
            )
        ].copy()

        candidates = candidates.sort_values(
            [
                "wickets_per_100_balls",
                "economy_rate",
                "legal_balls",
            ],
            ascending=[False, True, False],
        )

        shortlist = candidates[
            [
                "year",
                "match_phase",
                "team",
                "bowler_name",
                "legal_balls",
                "overs_bowled",
                "runs_conceded",
                "wickets",
                "economy_rate",
                "bowling_dot_pct",
                "boundary_balls_per_100_legal_balls",
                "wides_conceded",
                "noball_extras_conceded",
            ]
        ].head(10)

        bowling_candidate_shortlists[(year, phase)] = shortlist

        print(
            f"\n{year} | {phase} | "
            f"Eligible candidates: {len(candidates)}"
        )

        display(shortlist.round(2))


2023 | Powerplay | Eligible candidates: 31


,year,match_phase,team,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
387,2023,Powerplay,Punjab Kings,Nathan Ellis,60,10.0,73,4,7.30,40.00,18.33,0,0
214,2023,Powerplay,Gujarat Titans,Mohammed Shami,264,44.0,330,17,7.50,55.68,19.70,16,1
74,2023,Powerplay,Rajasthan Royals,Trent Boult,156,26.0,177,10,6.81,55.77,19.23,6,0
186,2023,Powerplay,Chennai Super Kings,Deepak Chahar,162,27.0,227,10,8.41,41.98,21.60,5,0
327,2023,Powerplay,Sunrisers Hyderabad,Marco Jansen,114,19.0,166,7,8.74,47.37,22.81,11,2
217,2023,Powerplay,Delhi Capitals,Anrich Nortje,84,14.0,123,5,8.79,51.19,27.38,4,2
47,2023,Powerplay,Delhi Capitals,Ishant Sharma,102,17.0,134,6,7.88,49.02,24.51,2,0
55,2023,Powerplay,Royal Challengers Bengaluru,Wayne Parnell,102,17.0,141,6,8.29,44.12,22.55,2,0
414,2023,Powerplay,Royal Challengers Bengaluru,Mohammed Siraj,180,30.0,178,10,5.93,63.33,16.67,15,0
253,2023,Powerplay,Punjab Kings,Kagiso Rabada,72,12.0,112,4,9.33,50.00,26.39,1,1



2023 | Middle | Eligible candidates: 39


,year,match_phase,team,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
246,2023,Middle,Gujarat Titans,Mohit Sharma,132,22.0,165,13,7.50,33.33,14.39,1,0
571,2023,Middle,Punjab Kings,Harpreet Brar,90,15.0,93,7,6.20,37.78,10.00,1,0
59,2023,Middle,Delhi Capitals,Mitchell Marsh,90,15.0,125,7,8.33,25.56,11.11,7,1
22,2023,Middle,Mumbai Indians,Piyush Chawla,288,48.0,394,20,8.21,30.21,15.62,5,0
15,2023,Middle,Royal Challengers Bengaluru,Karn Sharma,117,19.5,201,8,10.31,29.91,22.22,2,0
42,2023,Middle,Chennai Super Kings,Ravindra Jadeja,330,55.0,398,20,7.24,32.12,10.91,11,0
361,2023,Middle,Gujarat Titans,Rashid Khan,252,42.0,324,15,7.71,32.54,14.68,4,0
404,2023,Middle,Gujarat Titans,Josh Little,84,14.0,118,5,8.43,35.71,17.86,2,0
9,2023,Middle,Rajasthan Royals,Ravichandran Ashwin,186,31.0,232,11,7.48,27.96,10.22,12,0
607,2023,Middle,Gujarat Titans,Noor Ahmad,210,35.0,278,12,7.94,32.86,13.81,10,1



2023 | Death | Eligible candidates: 24


...

,year,match_phase,team,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
274,2024,Powerplay,Gujarat Titans,Sandeep Warrier,60,10.0,104,6,10.40,46.67,28.33,3,1
432,2024,Powerplay,Mumbai Indians,Nuwan Thushara,78,13.0,124,6,9.54,41.03,26.92,8,0
439,2024,Powerplay,Rajasthan Royals,Nandre Burger,60,10.0,99,4,9.90,48.33,30.00,6,0
50,2024,Powerplay,Delhi Capitals,Ishant Sharma,108,18.0,167,7,9.28,47.22,24.07,16,0
324,2024,Powerplay,Rajasthan Royals,Avesh Khan,78,13.0,134,5,10.31,29.49,26.92,1,0
402,2024,Powerplay,Delhi Capitals,Mukesh Kumar,66,11.0,113,4,10.27,40.91,33.33,0,1
77,2024,Powerplay,Rajasthan Royals,Trent Boult,228,38.0,267,12,7.03,49.12,17.98,7,0
138,2024,Powerplay,Gujarat Titans,Umesh Yadav,102,17.0,149,5,8.76,49.02,25.49,1,0
300,2024,Powerplay,Mumbai Indians,Jasprit Bumrah,126,21.0,126,6,6.00,61.11,15.08,11,1
283,2024,Powerplay,Mumbai Indians,Gerald Coetzee,66,11.0,119,3,10.82,45.45,28.79,10,1



2024 | Middle | Eligible candidates: 41


,year,match_phase,team,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
615,2024,Middle,Chennai Super Kings,Matheesha Pathirana,66,11.0,71,8,6.45,45.45,9.09,12,0
305,2024,Middle,Punjab Kings,Sam Curran,60,10.0,95,6,9.50,28.33,21.67,0,0
358,2024,Middle,Lucknow Super Giants,Naveen-ul-Haq,60,10.0,83,5,8.30,35.00,15.00,3,0
416,2024,Middle,Royal Challengers Bengaluru,Mohammed Siraj,60,10.0,93,5,9.30,23.33,15.00,7,0
563,2024,Middle,Delhi Capitals,Rasikh Salam,78,13.0,137,6,10.54,24.36,21.79,5,0
479,2024,Middle,Sunrisers Hyderabad,Mayank Markande,114,19.0,202,8,10.63,21.05,18.42,7,1
444,2024,Middle,Gujarat Titans,Sai Kishore,60,10.0,72,4,7.20,38.33,11.67,2,0
108,2024,Middle,Royal Challengers Bengaluru,Glenn Maxwell,60,10.0,74,4,7.40,38.33,13.33,0,0
268,2024,Middle,Delhi Capitals,Kuldeep Yadav,204,34.0,272,12,8.00,32.84,13.73,7,1
222,2024,Middle,Sunrisers Hyderabad,Pat Cummins,138,23.0,175,8,7.61,35.51,15.22,2,0



2024 | Death | Eligible candidates: 25


,year,match_phase,team,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
152,2024,Death,Punjab Kings,Harshal Patel,126,21.00,218,16,10.38,28.57,21.43,13,0
298,2024,Death,Mumbai Indians,Jasprit Bumrah,101,16.83,109,11,6.48,34.65,9.90,2,0
427,2024,Death,Delhi Capitals,Khaleel Ahmed,66,11.00,134,7,12.18,34.85,27.27,9,1
380,2024,Death,Chennai Super Kings,Tushar Deshpande,96,16.00,159,10,9.94,33.33,20.83,14,1
509,2024,Death,Punjab Kings,Arshdeep Singh,98,16.33,198,10,12.12,32.65,28.57,14,0
281,2024,Death,Mumbai Indians,Gerald Coetzee,69,11.50,107,7,9.30,31.88,17.39,7,0
552,2024,Death,Royal Challengers Bengaluru,Yash Dayal,91,15.17,147,9,9.69,34.07,21.98,11,3
304,2024,Death,Punjab Kings,Sam Curran,72,12.00,156,7,13.00,18.06,30.56,4,1
400,2024,Death,Delhi Capitals,Mukesh Kumar,93,15.50,175,9,11.29,26.88,29.03,3,1
181,2024,Death,Rajasthan Royals,Sandeep Sharma,84,14.00,137,8,9.79,22.62,20.24,2,0


Incumbent vs External Candidate Comparison

In [40]:
# Step 19.1: Select the top three external candidates
# for every batting and bowling phase in both seasons.

TOP_N_CANDIDATES = 3

# ---------------------------------------------------------
# BATTING CANDIDATES
# ---------------------------------------------------------

batting_targets = []

for year in [2023, 2024]:
    for phase in ["Powerplay", "Middle", "Death"]:

        candidates = batting_candidates.loc[
            batting_candidates["year"].eq(year)
            & batting_candidates["match_phase"].eq(phase)
        ].copy()

        # Apply the same minimum opportunity thresholds
        # used in the initial scouting exercise.
        min_balls = {
            "Powerplay": MIN_PP_BALLS,
            "Middle": MIN_MIDDLE_BALLS,
            "Death": MIN_DEATH_BALLS,
        }[phase]

        candidates = candidates.loc[
            candidates["balls_faced"].ge(min_balls)
        ]

        # Initial ranking: scoring rate, then run volume,
        # then balls faced.
        candidates = candidates.sort_values(
            ["strike_rate", "runs", "balls_faced"],
            ascending=[False, False, False],
        )

        top_candidates = candidates.head(TOP_N_CANDIDATES)

        for _, row in top_candidates.iterrows():
            batting_targets.append(
                (year, phase, row["batter_name"])
            )

        print(f"\n{year} | {phase} | Batting candidates")

        display(
            top_candidates[
                [
                    "year",
                    "match_phase",
                    "team",
                    "batter_name",
                    "runs",
                    "balls_faced",
                    "strike_rate",
                    "dot_ball_pct",
                    "boundary_ball_pct",
                ]
            ].round(2)
        )


# ---------------------------------------------------------
# BOWLING CANDIDATES
# ---------------------------------------------------------

bowling_targets = []

for year in [2023, 2024]:
    for phase in ["Powerplay", "Middle", "Death"]:

        candidates = bowling_candidates.loc[
            bowling_candidates["year"].eq(year)
            & bowling_candidates["match_phase"].eq(phase)
        ].copy()

        candidates = candidates.loc[
            candidates["legal_balls"].ge(MIN_BOWLING_BALLS)
        ]

        # Initial ranking: wicket-taking rate, followed by
        # economy and workload.
        candidates = candidates.sort_values(
            [
                "wickets_per_100_balls",
                "economy_rate",
                "legal_balls",
            ],
            ascending=[False, True, False],
        )

        top_candidates = candidates.head(TOP_N_CANDIDATES)

        for _, row in top_candidates.iterrows():
            bowling_targets.append(
                (year, phase, row["bowler_name"])
            )

        print(f"\n{year} | {phase} | Bowling candidates")

        display(
            top_candidates[
                [
                    "year",
                    "match_phase",
                    "team",
                    "bowler_name",
                    "legal_balls",
                    "overs_bowled",
                    "runs_conceded",
                    "wickets",
                    "wickets_per_100_balls",
                    "economy_rate",
                    "bowling_dot_pct",
                    "boundary_balls_per_100_legal_balls",
                    "wides_conceded",
                    "noball_extras_conceded",
                ]
            ].round(2)
        )


# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

print("\nTotal batting candidate-role combinations:", len(batting_targets))
print("Total bowling candidate-role combinations:", len(bowling_targets))

assert len(batting_targets) == 18, (
    "Expected 18 batting candidate-role combinations. "
    "Check whether any season-phase group has fewer than three eligible candidates."
)

assert len(bowling_targets) == 18, (
    "Expected 18 bowling candidate-role combinations. "
    "Check whether any season-phase group has fewer than three eligible candidates."
)


2023 | Powerplay | Batting candidates


,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
99,2023,Powerplay,Chennai Super Kings,Ajinkya Rahane,125,60,208.33,23.33,31.67
620,2023,Powerplay,Rajasthan Royals,Yashasvi Jaiswal,361,206,175.24,46.60,34.47
551,2023,Powerplay,Mumbai Indians,Cameron Green,167,96,173.96,31.25,30.21



2023 | Middle | Batting candidates


,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
260,2023,Middle,Mumbai Indians,Suryakumar Yadav,450,246,182.93,25.20,28.05
160,2023,Middle,Royal Challengers Bengaluru,Glenn Maxwell,249,141,176.60,29.08,23.40
345,2023,Middle,Lucknow Super Giants,Nicholas Pooran,176,104,169.23,37.50,26.92



2023 | Death | Batting candidates


...


2024 | Middle | Bowling candidates


,year,match_phase,team,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,wickets_per_100_balls,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
615,2024,Middle,Chennai Super Kings,Matheesha Pathirana,66,11.0,71,8,12.12,6.45,45.45,9.09,12,0
305,2024,Middle,Punjab Kings,Sam Curran,60,10.0,95,6,10.00,9.50,28.33,21.67,0,0
358,2024,Middle,Lucknow Super Giants,Naveen-ul-Haq,60,10.0,83,5,8.33,8.30,35.00,15.00,3,0



2024 | Death | Bowling candidates


,year,match_phase,team,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,wickets_per_100_balls,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
152,2024,Death,Punjab Kings,Harshal Patel,126,21.00,218,16,12.70,10.38,28.57,21.43,13,0
298,2024,Death,Mumbai Indians,Jasprit Bumrah,101,16.83,109,11,10.89,6.48,34.65,9.90,2,0
427,2024,Death,Delhi Capitals,Khaleel Ahmed,66,11.00,134,7,10.61,12.18,34.85,27.27,9,1



Total batting candidate-role combinations: 18
Total bowling candidate-role combinations: 18


In [41]:
batting_comparison_rows = []

for year, phase, candidate_name in batting_targets:

    # KKR's incumbents in the same season and phase.
    incumbents = kkr_batting_phase.loc[
        kkr_batting_phase["year"].eq(year)
        & kkr_batting_phase["match_phase"].eq(phase)
    ].copy()

    incumbents = (
        incumbents
        .sort_values(
            ["balls_faced", "runs"],
            ascending=[False, False],
        )
        .head(3)
    )

    # The selected external candidate in the same season and phase.
    candidate = batting_candidates.loc[
        batting_candidates["year"].eq(year)
        & batting_candidates["match_phase"].eq(phase)
        & batting_candidates["batter_name"].eq(candidate_name)
    ].copy()

    if candidate.empty:
        print(
            f"WARNING: {candidate_name} not found for "
            f"{year} {phase}."
        )
        continue

    # Standardise columns so candidate and incumbents can be compared.
    candidate_output = candidate[
        [
            "year",
            "match_phase",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
        ]
    ].copy()

    candidate_output["player_status"] = "External candidate"

    incumbent_output = incumbents[
        [
            "year",
            "match_phase",
            "batter_name",
            "runs",
            "balls_faced",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
        ]
    ].copy()

    incumbent_output["player_status"] = "KKR incumbent"

    comparison = pd.concat(
        [candidate_output, incumbent_output],
        ignore_index=True,
    )

    comparison["selected_candidate"] = candidate_name

    batting_comparison_rows.append(comparison)

batting_incumbent_comparison = pd.concat(
    batting_comparison_rows,
    ignore_index=True,
) if batting_comparison_rows else pd.DataFrame()

batting_incumbent_comparison = batting_incumbent_comparison[
    [
        "year",
        "match_phase",
        "selected_candidate",
        "player_status",
        "batter_name",
        "runs",
        "balls_faced",
        "strike_rate",
        "dot_ball_pct",
        "boundary_ball_pct",
    ]
]

display(
    batting_incumbent_comparison
    .sort_values(
        ["year", "match_phase", "player_status", "balls_faced"],
        ascending=[True, True, True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

,year,match_phase,selected_candidate,player_status,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
0,2023,Death,Vijay Shankar,External candidate,Vijay Shankar,122,47,259.57,17.02,38.30
1,2023,Death,Liam Livingstone,External candidate,Liam Livingstone,103,41,251.22,34.15,39.02
2,2023,Death,Glenn Maxwell,External candidate,Glenn Maxwell,68,31,219.35,35.48,35.48
3,2023,Death,Vijay Shankar,KKR incumbent,Rinku Singh,280,150,186.67,32.00,27.33
4,2023,Death,Liam Livingstone,KKR incumbent,Rinku Singh,280,150,186.67,32.00,27.33
...,...,...,...,...,...,...,...,...,...,...
67,2024,Powerplay,Travis Head,KKR incumbent,Phil Salt,296,160,185.00,40.62,33.12
68,2024,Powerplay,Abhishek Sharma,KKR incumbent,Phil Salt,296,160,185.00,40.62,33.12
69,2024,Powerplay,Jake Fraser-McGurk,KKR incumbent,Venkatesh Iyer,111,50,222.00,34.00,44.00
70,2024,Powerplay,Travis Head,KKR incumbent,Venkatesh Iyer,111,50,222.00,34.00,44.00


In [42]:
bowling_comparison_rows = []

for year, phase, candidate_name in bowling_targets:

    # KKR's incumbents in the same season and phase.
    incumbents = kkr_bowling_phase.loc[
        kkr_bowling_phase["year"].eq(year)
        & kkr_bowling_phase["match_phase"].eq(phase)
    ].copy()

    incumbents = (
        incumbents
        .sort_values(
            ["legal_balls", "wickets"],
            ascending=[False, False],
        )
        .head(3)
    )

    # The selected external candidate in the same season and phase.
    candidate = bowling_candidates.loc[
        bowling_candidates["year"].eq(year)
        & bowling_candidates["match_phase"].eq(phase)
        & bowling_candidates["bowler_name"].eq(candidate_name)
    ].copy()

    if candidate.empty:
        print(
            f"WARNING: {candidate_name} not found for "
            f"{year} {phase}."
        )
        continue

    columns = [
        "year",
        "match_phase",
        "bowler_name",
        "legal_balls",
        "overs_bowled",
        "runs_conceded",
        "wickets",
        "economy_rate",
        "bowling_dot_pct",
        "boundary_balls_per_100_legal_balls",
        "wides_conceded",
        "noball_extras_conceded",
    ]

    candidate_output = candidate[columns].copy()
    candidate_output["player_status"] = "External candidate"

    incumbent_output = incumbents[columns].copy()
    incumbent_output["player_status"] = "KKR incumbent"

    comparison = pd.concat(
        [candidate_output, incumbent_output],
        ignore_index=True,
    )

    comparison["selected_candidate"] = candidate_name

    bowling_comparison_rows.append(comparison)

bowling_incumbent_comparison = pd.concat(
    bowling_comparison_rows,
    ignore_index=True,
) if bowling_comparison_rows else pd.DataFrame()

bowling_incumbent_comparison = bowling_incumbent_comparison[
    [
        "year",
        "match_phase",
        "selected_candidate",
        "player_status",
        "bowler_name",
        "legal_balls",
        "overs_bowled",
        "runs_conceded",
        "wickets",
        "economy_rate",
        "bowling_dot_pct",
        "boundary_balls_per_100_legal_balls",
        "wides_conceded",
        "noball_extras_conceded",
    ]
]

display(
    bowling_incumbent_comparison
    .sort_values(
        ["year", "match_phase", "player_status", "legal_balls"],
        ascending=[True, True, True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

,year,match_phase,selected_candidate,player_status,bowler_name,legal_balls,overs_bowled,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
0,2023,Death,Mohit Sharma,External candidate,Mohit Sharma,115,19.17,168,14,8.77,30.43,17.39,2,0
1,2023,Death,Yuzvendra Chahal,External candidate,Yuzvendra Chahal,84,14.00,112,11,8.00,35.71,14.29,3,0
2,2023,Death,Rashid Khan,External candidate,Rashid Khan,78,13.00,114,10,8.77,33.33,16.67,1,0
3,2023,Death,Yuzvendra Chahal,KKR incumbent,Varun Chakravarthy,76,12.67,106,4,8.37,31.58,14.47,3,0
4,2023,Death,Rashid Khan,KKR incumbent,Varun Chakravarthy,76,12.67,106,4,8.37,31.58,14.47,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2024,Powerplay,Nuwan Thushara,KKR incumbent,Vaibhav Arora,144,24.00,207,9,8.62,43.75,20.83,14,5
68,2024,Powerplay,Nandre Burger,KKR incumbent,Vaibhav Arora,144,24.00,207,9,8.62,43.75,20.83,14,5
69,2024,Powerplay,Sandeep Warrier,KKR incumbent,Harshit Rana,84,14.00,139,4,9.93,44.05,26.19,4,0
70,2024,Powerplay,Nuwan Thushara,KKR incumbent,Harshit Rana,84,14.00,139,4,9.93,44.05,26.19,4,0


In [44]:
# Inspect the selected candidates without repeating incumbent rows.
print("Batting targets:", len(batting_targets))
print("Bowling targets:", len(bowling_targets))

# --------------------------------------------------
# Batting candidates
# --------------------------------------------------
batting_candidate_rows = []

for year, phase, name in batting_targets:
    match = batting_candidates.loc[
        (batting_candidates["year"] == year)
        & (batting_candidates["match_phase"] == phase)
        & (batting_candidates["batter_name"] == name)
    ]

    if len(match) != 1:
        print(f"Warning: expected 1 batting record for {year}, {phase}, {name}; found {len(match)}")

    batting_candidate_rows.append(match)

batting_candidate_summary = pd.concat(
    batting_candidate_rows, ignore_index=True
)

display(
    batting_candidate_summary[
        [
            "year", "match_phase", "team", "batter_name",
            "runs", "balls_faced", "strike_rate",
            "dot_ball_pct", "boundary_ball_pct"
        ]
    ].sort_values(
        ["year", "match_phase", "strike_rate"],
        ascending=[True, True, False]
    ).reset_index(drop=True)
)


# --------------------------------------------------
# Bowling candidates
# --------------------------------------------------
bowling_candidate_rows = []

for year, phase, name in bowling_targets:
    match = bowling_candidates.loc[
        (bowling_candidates["year"] == year)
        & (bowling_candidates["match_phase"] == phase)
        & (bowling_candidates["bowler_name"] == name)
    ]

    if len(match) != 1:
        print(f"Warning: expected 1 bowling record for {year}, {phase}, {name}; found {len(match)}")

    bowling_candidate_rows.append(match)

bowling_candidate_summary = pd.concat(
    bowling_candidate_rows, ignore_index=True
)

display(
    bowling_candidate_summary[
        [
            "year", "match_phase", "team", "bowler_name",
            "legal_balls", "runs_conceded", "wickets",
            "economy_rate", "bowling_dot_pct",
            "boundary_balls_per_100_legal_balls",
            "wides_conceded", "noball_extras_conceded"
        ]
    ].sort_values(
        ["year", "match_phase", "economy_rate"],
        ascending=[True, True, True]
    ).reset_index(drop=True)
)

Batting targets: 18
Bowling targets: 18


,year,match_phase,team,batter_name,runs,balls_faced,strike_rate,dot_ball_pct,boundary_ball_pct
0,2023,Death,Gujarat Titans,Vijay Shankar,122,47,259.574468,17.021277,38.297872
1,2023,Death,Punjab Kings,Liam Livingstone,103,41,251.219512,34.146341,39.024390
2,2023,Death,Royal Challengers Bengaluru,Glenn Maxwell,68,31,219.354839,35.483871,35.483871
3,2023,Middle,Mumbai Indians,Suryakumar Yadav,450,246,182.926829,25.203252,28.048780
4,2023,Middle,Royal Challengers Bengaluru,Glenn Maxwell,249,141,176.595745,29.078014,23.404255
5,2023,Middle,Lucknow Super Giants,Nicholas Pooran,176,104,169.230769,37.500000,26.923077
6,2023,Powerplay,Chennai Super Kings,Ajinkya Rahane,125,60,208.333333,23.333333,31.666667
7,2023,Powerplay,Rajasthan Royals,Yashasvi Jaiswal,361,206,175.242718,46.601942,34.466019
8,2023,Powerplay,Mumbai Indians,Cameron Green,167,96,173.958333,31.250000,30.208333
9,2024,Death,Delhi Capitals,Tristan Stubbs,252,96,262.500000,11.458333,39.583333


,year,match_phase,team,bowler_name,legal_balls,runs_conceded,wickets,economy_rate,bowling_dot_pct,boundary_balls_per_100_legal_balls,wides_conceded,noball_extras_conceded
0,2023,Death,Rajasthan Royals,Yuzvendra Chahal,84,112,11,8.000000,35.714286,14.285714,3,0
1,2023,Death,Gujarat Titans,Mohit Sharma,115,168,14,8.765217,30.434783,17.391304,2,0
2,2023,Death,Gujarat Titans,Rashid Khan,78,114,10,8.769231,33.333333,16.666667,1,0
3,2023,Middle,Punjab Kings,Harpreet Brar,90,93,7,6.200000,37.777778,10.000000,1,0
4,2023,Middle,Gujarat Titans,Mohit Sharma,132,165,13,7.500000,33.333333,14.393939,1,0
5,2023,Middle,Delhi Capitals,Mitchell Marsh,90,125,7,8.333333,25.555556,11.111111,7,1
6,2023,Powerplay,Rajasthan Royals,Trent Boult,156,177,10,6.807692,55.769231,19.230769,6,0
7,2023,Powerplay,Punjab Kings,Nathan Ellis,60,73,4,7.300000,40.000000,18.333333,0,0
8,2023,Powerplay,Gujarat Titans,Mohammed Shami,264,330,17,7.500000,55.681818,19.696970,16,1
9,2024,Death,Mumbai Indians,Jasprit Bumrah,101,109,11,6.475248,34.653465,9.900990,2,0


In [45]:
# Validate the scouting shortlist.

batting_shortlist_check = (
    batting_candidate_summary
    .groupby(["year", "match_phase"])
    .agg(
        candidate_count=("batter_name", "nunique"),
        total_balls=("balls_faced", "sum")
    )
    .reset_index()
)

bowling_shortlist_check = (
    bowling_candidate_summary
    .groupby(["year", "match_phase"])
    .agg(
        candidate_count=("bowler_name", "nunique"),
        total_legal_balls=("legal_balls", "sum")
    )
    .reset_index()
)

print("Batting shortlist validation")
display(batting_shortlist_check)

print("Bowling shortlist validation")
display(bowling_shortlist_check)

assert batting_shortlist_check["candidate_count"].eq(3).all()
assert bowling_shortlist_check["candidate_count"].eq(3).all()

print("All six phase-season groups contain three unique candidates per role.")

Batting shortlist validation


,year,match_phase,candidate_count,total_balls
0,2023,Death,3,119
1,2023,Middle,3,491
2,2023,Powerplay,3,362
3,2024,Death,3,216
4,2024,Middle,3,303
5,2024,Powerplay,3,468


Bowling shortlist validation


,year,match_phase,candidate_count,total_legal_balls
0,2023,Death,3,277
1,2023,Middle,3,312
2,2023,Powerplay,3,480
3,2024,Death,3,293
4,2024,Middle,3,186
5,2024,Powerplay,3,198


All six phase-season groups contain three unique candidates per role.


In [46]:
# ============================================================
# CELL 20.1 — External candidates vs KKR incumbents
# ============================================================

# For each external candidate, compare against KKR's three
# highest-workload players in the same season and phase.
#
# This is a descriptive comparison, not a final recruitment
# ranking. Workload and rate statistics are shown separately.

batting_comparisons = []

for _, candidate in batting_candidate_summary.iterrows():

    year = candidate["year"]
    phase = candidate["match_phase"]

    incumbents = (
        kkr_batting_phase.loc[
            (kkr_batting_phase["year"] == year)
            & (kkr_batting_phase["match_phase"] == phase)
        ]
        .sort_values(
            ["balls_faced", "runs"],
            ascending=[False, False]
        )
        .head(3)
    )

    for _, incumbent in incumbents.iterrows():

        batting_comparisons.append({
            "year": year,
            "phase": phase,
            "external_candidate": candidate["batter_name"],
            "candidate_team": candidate["team"],
            "kkr_incumbent": incumbent["batter_name"],

            "candidate_balls": candidate["balls_faced"],
            "incumbent_balls": incumbent["balls_faced"],

            "candidate_runs": candidate["runs"],
            "incumbent_runs": incumbent["runs"],

            "candidate_SR": candidate["strike_rate"],
            "incumbent_SR": incumbent["strike_rate"],
            "SR_difference": (
                candidate["strike_rate"] - incumbent["strike_rate"]
            ),

            "candidate_dot_pct": candidate["dot_ball_pct"],
            "incumbent_dot_pct": incumbent["dot_ball_pct"],
            "dot_pct_difference": (
                candidate["dot_ball_pct"] - incumbent["dot_ball_pct"]
            ),

            "candidate_boundary_pct": candidate["boundary_ball_pct"],
            "incumbent_boundary_pct": incumbent["boundary_ball_pct"],
            "boundary_pct_difference": (
                candidate["boundary_ball_pct"]
                - incumbent["boundary_ball_pct"]
            ),
        })


batting_comparison_df = pd.DataFrame(batting_comparisons)

print("External batting candidates vs KKR incumbents")
display(
    batting_comparison_df.round(2).sort_values(
        ["year", "phase", "external_candidate", "kkr_incumbent"]
    ).reset_index(drop=True)
)


# ============================================================
# Bowling comparisons
# ============================================================

bowling_comparisons = []

for _, candidate in bowling_candidate_summary.iterrows():

    year = candidate["year"]
    phase = candidate["match_phase"]

    incumbents = (
        kkr_bowling_phase.loc[
            (kkr_bowling_phase["year"] == year)
            & (kkr_bowling_phase["match_phase"] == phase)
        ]
        .sort_values(
            ["legal_balls", "wickets"],
            ascending=[False, False]
        )
        .head(3)
    )

    for _, incumbent in incumbents.iterrows():

        candidate_wickets_per_100 = (
            100 * candidate["wickets"] / candidate["legal_balls"]
            if candidate["legal_balls"] > 0 else float("nan")
        )

        incumbent_wickets_per_100 = (
            100 * incumbent["wickets"] / incumbent["legal_balls"]
            if incumbent["legal_balls"] > 0 else float("nan")
        )

        bowling_comparisons.append({
            "year": year,
            "phase": phase,
            "external_candidate": candidate["bowler_name"],
            "candidate_team": candidate["team"],
            "kkr_incumbent": incumbent["bowler_name"],

            "candidate_legal_balls": candidate["legal_balls"],
            "incumbent_legal_balls": incumbent["legal_balls"],

            "candidate_wickets": candidate["wickets"],
            "incumbent_wickets": incumbent["wickets"],

            "candidate_wickets_per_100": candidate_wickets_per_100,
            "incumbent_wickets_per_100": incumbent_wickets_per_100,
            "wicket_rate_difference": (
                candidate_wickets_per_100 - incumbent_wickets_per_100
            ),

            "candidate_economy": candidate["economy_rate"],
            "incumbent_economy": incumbent["economy_rate"],
            "economy_difference": (
                candidate["economy_rate"] - incumbent["economy_rate"]
            ),

            "candidate_dot_pct": candidate["bowling_dot_pct"],
            "incumbent_dot_pct": incumbent["bowling_dot_pct"],
            "dot_pct_difference": (
                candidate["bowling_dot_pct"] - incumbent["bowling_dot_pct"]
            ),

            "candidate_boundary_freq": (
                candidate["boundary_balls_per_100_legal_balls"]
            ),
            "incumbent_boundary_freq": (
                incumbent["boundary_balls_per_100_legal_balls"]
            ),
            "boundary_freq_difference": (
                candidate["boundary_balls_per_100_legal_balls"]
                - incumbent["boundary_balls_per_100_legal_balls"]
            ),

            "candidate_wides": candidate["wides_conceded"],
            "incumbent_wides": incumbent["wides_conceded"],

            "candidate_noball_extras": candidate["noball_extras_conceded"],
            "incumbent_noball_extras": incumbent["noball_extras_conceded"],
        })


bowling_comparison_df = pd.DataFrame(bowling_comparisons)

print("External bowling candidates vs KKR incumbents")
display(
    bowling_comparison_df.round(2).sort_values(
        ["year", "phase", "external_candidate", "kkr_incumbent"]
    ).reset_index(drop=True)
)


# ============================================================
# Validation
# ============================================================

assert not batting_comparison_df.empty
assert not bowling_comparison_df.empty

assert batting_comparison_df[
    ["external_candidate", "kkr_incumbent", "year", "phase"]
].notna().all().all()

assert bowling_comparison_df[
    ["external_candidate", "kkr_incumbent", "year", "phase"]
].notna().all().all()

print("Comparison tables generated and validated.")

External batting candidates vs KKR incumbents


,year,phase,external_candidate,candidate_team,kkr_incumbent,candidate_balls,incumbent_balls,candidate_runs,incumbent_runs,candidate_SR,incumbent_SR,SR_difference,candidate_dot_pct,incumbent_dot_pct,dot_pct_difference,candidate_boundary_pct,incumbent_boundary_pct,boundary_pct_difference
0,2023,Death,Glenn Maxwell,Royal Challengers Bengaluru,Andre Russell,31,82,68,134,219.35,163.41,55.94,35.48,43.90,-8.42,35.48,24.39,11.09
1,2023,Death,Glenn Maxwell,Royal Challengers Bengaluru,Rinku Singh,31,150,68,280,219.35,186.67,32.69,35.48,32.00,3.48,35.48,27.33,8.15
2,2023,Death,Glenn Maxwell,Royal Challengers Bengaluru,Shardul Thakur,31,39,68,52,219.35,133.33,86.02,35.48,35.90,-0.41,35.48,20.51,14.97
3,2023,Death,Liam Livingstone,Punjab Kings,Andre Russell,41,82,103,134,251.22,163.41,87.80,34.15,43.90,-9.76,39.02,24.39,14.63
4,2023,Death,Liam Livingstone,Punjab Kings,Rinku Singh,41,150,103,280,251.22,186.67,64.55,34.15,32.00,2.15,39.02,27.33,11.69
5,2023,Death,Liam Livingstone,Punjab Kings,Shardul Thakur,41,39,103,52,251.22,133.33,117.89,34.15,35.90,-1.75,39.02,20.51,18.51
6,2023,Death,Vijay Shankar,Gujarat Titans,Andre Russell,47,82,122,134,259.57,163.41,96.16,17.02,43.90,-26.88,38.30,24.39,13.91
7,2023,Death,Vijay Shankar,Gujarat Titans,Rinku Singh,47,150,122,280,259.57,186.67,72.91,17.02,32.00,-14.98,38.30,27.33,10.96
8,2023,Death,Vijay Shankar,Gujarat Titans,Shardul Thakur,47,39,122,52,259.57,133.33,126.24,17.02,35.90,-18.88,38.30,20.51,17.79
9,2023,Middle,Glenn Maxwell,Royal Challengers Bengaluru,Nitish Rana,141,193,249,269,176.60,139.38,37.22,29.08,33.68,-4.60,23.40,19.17,4.23


External bowling candidates vs KKR incumbents


,year,phase,external_candidate,candidate_team,kkr_incumbent,candidate_legal_balls,incumbent_legal_balls,candidate_wickets,incumbent_wickets,candidate_wickets_per_100,...,candidate_dot_pct,incumbent_dot_pct,dot_pct_difference,candidate_boundary_freq,incumbent_boundary_freq,boundary_freq_difference,candidate_wides,incumbent_wides,candidate_noball_extras,incumbent_noball_extras
0,2023,Death,Mohit Sharma,Gujarat Titans,Shardul Thakur,115,47,14,3,12.17,...,30.43,23.40,7.03,17.39,25.53,-8.14,2,3,0,0
1,2023,Death,Mohit Sharma,Gujarat Titans,Sunil Narine,115,48,14,3,12.17,...,30.43,25.00,5.43,17.39,14.58,2.81,2,2,0,0
2,2023,Death,Mohit Sharma,Gujarat Titans,Varun Chakravarthy,115,76,14,4,12.17,...,30.43,31.58,-1.14,17.39,14.47,2.92,2,3,0,0
3,2023,Death,Rashid Khan,Gujarat Titans,Shardul Thakur,78,47,10,3,12.82,...,33.33,23.40,9.93,16.67,25.53,-8.87,1,3,0,0
4,2023,Death,Rashid Khan,Gujarat Titans,Sunil Narine,78,48,10,3,12.82,...,33.33,25.00,8.33,16.67,14.58,2.08,1,2,0,0
5,2023,Death,Rashid Khan,Gujarat Titans,Varun Chakravarthy,78,76,10,4,12.82,...,33.33,31.58,1.75,16.67,14.47,2.19,1,3,0,0
6,2023,Death,Yuzvendra Chahal,Rajasthan Royals,Shardul Thakur,84,47,11,3,13.10,...,35.71,23.40,12.31,14.29,25.53,-11.25,3,3,0,0
7,2023,Death,Yuzvendra Chahal,Rajasthan Royals,Sunil Narine,84,48,11,3,13.10,...,35.71,25.00,10.71,14.29,14.58,-0.30,3,2,0,0
8,2023,Death,Yuzvendra Chahal,Rajasthan Royals,Varun Chakravarthy,84,76,11,4,13.10,...,35.71,31.58,4.14,14.29,14.47,-0.19,3,3,0,0
9,2023,Middle,Harpreet Brar,Punjab Kings,Sunil Narine,90,198,7,6,7.78,...,37.78,26.26,11.52,10.00,10.61,-0.61,1,3,0,0


Comparison tables generated and validated.


In [47]:
# ============================================================
# CELL 20.2 — Concise scouting shortlist
# ============================================================

batting_shortlist = batting_candidate_summary[
    [
        "year",
        "match_phase",
        "batter_name",
        "team",
        "runs",
        "balls_faced",
        "strike_rate",
        "dot_ball_pct",
        "boundary_ball_pct",
    ]
].copy()

batting_shortlist = batting_shortlist.rename(
    columns={
        "match_phase": "phase",
        "batter_name": "candidate",
        "runs": "runs_scored",
        "balls_faced": "balls",
        "strike_rate": "strike_rate",
        "dot_ball_pct": "dot_pct",
        "boundary_ball_pct": "boundary_pct",
    }
)

print("BATTING SHORTLIST")
display(
    batting_shortlist.sort_values(
        ["year", "phase", "strike_rate"],
        ascending=[True, True, False],
    ).round(2).reset_index(drop=True)
)


bowling_shortlist = bowling_candidate_summary[
    [
        "year",
        "match_phase",
        "bowler_name",
        "team",
        "legal_balls",
        "runs_conceded",
        "wickets",
        "economy_rate",
        "bowling_dot_pct",
        "boundary_balls_per_100_legal_balls",
        "wides_conceded",
        "noball_extras_conceded",
    ]
].copy()

bowling_shortlist = bowling_shortlist.rename(
    columns={
        "match_phase": "phase",
        "bowler_name": "candidate",
        "legal_balls": "legal_balls",
        "runs_conceded": "runs_conceded",
        "economy_rate": "economy",
        "bowling_dot_pct": "dot_pct",
        "boundary_balls_per_100_legal_balls": "boundary_balls_per_100",
        "noball_extras_conceded": "no_ball_extras",
    }
)

print("BOWLING SHORTLIST")
display(
    bowling_shortlist.sort_values(
        ["year", "phase", "wickets", "economy"],
        ascending=[True, True, False, True],
    ).round(2).reset_index(drop=True)
)

BATTING SHORTLIST


,year,phase,candidate,team,runs_scored,balls,strike_rate,dot_pct,boundary_pct
0,2023,Death,Vijay Shankar,Gujarat Titans,122,47,259.57,17.02,38.30
1,2023,Death,Liam Livingstone,Punjab Kings,103,41,251.22,34.15,39.02
2,2023,Death,Glenn Maxwell,Royal Challengers Bengaluru,68,31,219.35,35.48,35.48
3,2023,Middle,Suryakumar Yadav,Mumbai Indians,450,246,182.93,25.20,28.05
4,2023,Middle,Glenn Maxwell,Royal Challengers Bengaluru,249,141,176.60,29.08,23.40
5,2023,Middle,Nicholas Pooran,Lucknow Super Giants,176,104,169.23,37.50,26.92
6,2023,Powerplay,Ajinkya Rahane,Chennai Super Kings,125,60,208.33,23.33,31.67
7,2023,Powerplay,Yashasvi Jaiswal,Rajasthan Royals,361,206,175.24,46.60,34.47
8,2023,Powerplay,Cameron Green,Mumbai Indians,167,96,173.96,31.25,30.21
9,2024,Death,Tristan Stubbs,Delhi Capitals,252,96,262.50,11.46,39.58


BOWLING SHORTLIST


,year,phase,candidate,team,legal_balls,runs_conceded,wickets,economy,dot_pct,boundary_balls_per_100,wides_conceded,no_ball_extras
0,2023,Death,Mohit Sharma,Gujarat Titans,115,168,14,8.77,30.43,17.39,2,0
1,2023,Death,Yuzvendra Chahal,Rajasthan Royals,84,112,11,8.00,35.71,14.29,3,0
2,2023,Death,Rashid Khan,Gujarat Titans,78,114,10,8.77,33.33,16.67,1,0
3,2023,Middle,Mohit Sharma,Gujarat Titans,132,165,13,7.50,33.33,14.39,1,0
4,2023,Middle,Harpreet Brar,Punjab Kings,90,93,7,6.20,37.78,10.00,1,0
5,2023,Middle,Mitchell Marsh,Delhi Capitals,90,125,7,8.33,25.56,11.11,7,1
6,2023,Powerplay,Mohammed Shami,Gujarat Titans,264,330,17,7.50,55.68,19.70,16,1
7,2023,Powerplay,Trent Boult,Rajasthan Royals,156,177,10,6.81,55.77,19.23,6,0
8,2023,Powerplay,Nathan Ellis,Punjab Kings,60,73,4,7.30,40.00,18.33,0,0
9,2024,Death,Harshal Patel,Punjab Kings,126,218,16,10.38,28.57,21.43,13,0


In [48]:
import pandas as pd

# ============================================================
# 1. BATTING SCOUTING DECISION TABLE
# ============================================================

batting_decisions = [
    {
        "year": 2023,
        "phase": "Powerplay",
        "candidate": "Yashasvi Jaiswal",
        "team": "Rajasthan Royals",
        "runs": 361,
        "balls": 206,
        "strike_rate": 175.24,
        "dot_pct": 46.60,
        "boundary_pct": 34.47,
        "kkr_benchmark": "Jason Roy",
        "kkr_benchmark_sr": 146.28,
        "advantage": "Higher scoring rate with substantial workload",
        "risk": "High dot-ball percentage; role and availability unknown",
        "classification": "Priority candidate",
    },
    {
        "year": 2023,
        "phase": "Middle",
        "candidate": "Suryakumar Yadav",
        "team": "Mumbai Indians",
        "runs": 450,
        "balls": 246,
        "strike_rate": 182.93,
        "dot_pct": 25.20,
        "boundary_pct": 28.05,
        "kkr_benchmark": "Venkatesh Iyer",
        "kkr_benchmark_sr": 160.27,
        "advantage": "Strong scoring rate, volume and dot-ball avoidance",
        "risk": "Role compatibility and availability not established",
        "classification": "Priority candidate",
    },
    {
        "year": 2023,
        "phase": "Death",
        "candidate": "Vijay Shankar",
        "team": "Gujarat Titans",
        "runs": 122,
        "balls": 47,
        "strike_rate": 259.57,
        "dot_pct": 17.02,
        "boundary_pct": 38.30,
        "kkr_benchmark": "Andre Russell",
        "kkr_benchmark_sr": 163.41,
        "advantage": "Exceptional historical scoring rate",
        "risk": "Only 47 balls; unusually high rate may not persist",
        "classification": "Monitor",
    },
    {
        "year": 2024,
        "phase": "Powerplay",
        "candidate": "Travis Head",
        "team": "Sunrisers Hyderabad",
        "runs": 402,
        "balls": 193,
        "strike_rate": 208.29,
        "dot_pct": 33.16,
        "boundary_pct": 38.34,
        "kkr_benchmark": "Phil Salt",
        "kkr_benchmark_sr": 185.00,
        "advantage": "High scoring rate supported by substantial volume",
        "risk": "KKR already had a productive opening combination",
        "classification": "Monitor",
    },
    {
        "year": 2024,
        "phase": "Middle",
        "candidate": "Rajat Patidar",
        "team": "Royal Challengers Bengaluru",
        "runs": 312,
        "balls": 172,
        "strike_rate": 181.40,
        "dot_pct": 28.49,
        "boundary_pct": 22.67,
        "kkr_benchmark": "Shreyas Iyer",
        "kkr_benchmark_sr": 140.13,
        "advantage": "Strong scoring rate with a meaningful workload",
        "risk": "Strike rate alone does not establish role equivalence",
        "classification": "Priority candidate",
    },
    {
        "year": 2024,
        "phase": "Death",
        "candidate": "Tristan Stubbs",
        "team": "Delhi Capitals",
        "runs": 252,
        "balls": 96,
        "strike_rate": 262.50,
        "dot_pct": 11.46,
        "boundary_pct": 39.58,
        "kkr_benchmark": "Andre Russell",
        "kkr_benchmark_sr": 210.94,
        "advantage": "Exceptional scoring rate, boundary frequency and dot-ball avoidance",
        "risk": "Limited sample; squad fit and availability unknown",
        "classification": "Priority candidate",
    },
]

batting_decision_df = pd.DataFrame(batting_decisions)

batting_decision_df["sr_advantage_vs_benchmark"] = (
    batting_decision_df["strike_rate"]
    - batting_decision_df["kkr_benchmark_sr"]
).round(2)

batting_decision_df = batting_decision_df[
    [
        "year",
        "phase",
        "candidate",
        "team",
        "runs",
        "balls",
        "strike_rate",
        "dot_pct",
        "boundary_pct",
        "kkr_benchmark",
        "kkr_benchmark_sr",
        "sr_advantage_vs_benchmark",
        "advantage",
        "risk",
        "classification",
    ]
]

print("BATTING SCOUTING DECISIONS")
display(
    batting_decision_df
    .sort_values(["year", "phase", "strike_rate"], ascending=[True, True, False])
    .reset_index(drop=True)
)


# ============================================================
# 2. BOWLING SCOUTING DECISION TABLE
# ============================================================

bowling_decisions = [
    {
        "year": 2023,
        "phase": "Powerplay",
        "candidate": "Mohammed Shami",
        "team": "Gujarat Titans",
        "legal_balls": 264,
        "runs_conceded": 330,
        "wickets": 17,
        "economy": 7.50,
        "dot_pct": 55.68,
        "boundary_balls_per_100": 19.70,
        "wides": 16,
        "no_ball_extras": 1,
        "kkr_benchmark": "Harshit Rana",
        "kkr_benchmark_economy": 7.93,
        "advantage": "Strong wicket output, economy and dot-ball pressure",
        "risk": "16 wides and one no-ball; historical season only",
        "classification": "Priority candidate",
    },
    {
        "year": 2023,
        "phase": "Powerplay",
        "candidate": "Trent Boult",
        "team": "Rajasthan Royals",
        "legal_balls": 156,
        "runs_conceded": 177,
        "wickets": 10,
        "economy": 6.81,
        "dot_pct": 55.77,
        "boundary_balls_per_100": 19.23,
        "wides": 6,
        "no_ball_extras": 0,
        "kkr_benchmark": "Harshit Rana",
        "kkr_benchmark_economy": 7.93,
        "advantage": "Excellent economy and dot-ball percentage",
        "risk": "Smaller workload than Shami; availability unknown",
        "classification": "Priority candidate",
    },
    {
        "year": 2023,
        "phase": "Middle",
        "candidate": "Harpreet Brar",
        "team": "Punjab Kings",
        "legal_balls": 90,
        "runs_conceded": 93,
        "wickets": 7,
        "economy": 6.20,
        "dot_pct": 37.78,
        "boundary_balls_per_100": 10.00,
        "wides": 1,
        "no_ball_extras": 0,
        "kkr_benchmark": "Sunil Narine",
        "kkr_benchmark_economy": 7.42,
        "advantage": "Strong containment and low boundary frequency",
        "risk": "Only 15 overs; wicket-taking sustainability uncertain",
        "classification": "Monitor",
    },
    {
        "year": 2024,
        "phase": "Middle",
        "candidate": "Matheesha Pathirana",
        "team": "Chennai Super Kings",
        "legal_balls": 66,
        "runs_conceded": 71,
        "wickets": 8,
        "economy": 6.45,
        "dot_pct": 45.45,
        "boundary_balls_per_100": 9.09,
        "wides": 12,
        "no_ball_extras": 0,
        "kkr_benchmark": "Varun Chakravarthy",
        "kkr_benchmark_economy": 7.68,
        "advantage": "High wicket output and excellent boundary suppression",
        "risk": "Only 11 overs and 12 wides; limited evidence",
        "classification": "Monitor",
    },
    {
        "year": 2024,
        "phase": "Death",
        "candidate": "Jasprit Bumrah",
        "team": "Mumbai Indians",
        "legal_balls": 101,
        "runs_conceded": 109,
        "wickets": 11,
        "economy": 6.48,
        "dot_pct": 34.65,
        "boundary_balls_per_100": 9.90,
        "wides": 2,
        "no_ball_extras": 0,
        "kkr_benchmark": "Andre Russell",
        "kkr_benchmark_economy": 9.64,
        "advantage": "Outstanding run prevention and boundary suppression",
        "risk": "Limited phase workload; availability and acquisition feasibility unknown",
        "classification": "Priority candidate",
    },
    {
        "year": 2024,
        "phase": "Death",
        "candidate": "Harshal Patel",
        "team": "Punjab Kings",
        "legal_balls": 126,
        "runs_conceded": 218,
        "wickets": 16,
        "economy": 10.38,
        "dot_pct": 28.57,
        "boundary_balls_per_100": 21.43,
        "wides": 13,
        "no_ball_extras": 0,
        "kkr_benchmark": "Andre Russell",
        "kkr_benchmark_economy": 9.64,
        "advantage": "High wicket output",
        "risk": "Expensive overs, frequent boundaries and 13 wides",
        "classification": "Monitor",
    },
]

bowling_decision_df = pd.DataFrame(bowling_decisions)

bowling_decision_df["economy_difference_vs_benchmark"] = (
    bowling_decision_df["economy"]
    - bowling_decision_df["kkr_benchmark_economy"]
).round(2)

bowling_decision_df = bowling_decision_df[
    [
        "year",
        "phase",
        "candidate",
        "team",
        "legal_balls",
        "runs_conceded",
        "wickets",
        "economy",
        "dot_pct",
        "boundary_balls_per_100",
        "wides",
        "no_ball_extras",
        "kkr_benchmark",
        "kkr_benchmark_economy",
        "economy_difference_vs_benchmark",
        "advantage",
        "risk",
        "classification",
    ]
]

print("BOWLING SCOUTING DECISIONS")
display(
    bowling_decision_df
    .sort_values(["year", "phase", "economy"])
    .reset_index(drop=True)
)

BATTING SCOUTING DECISIONS


,year,phase,candidate,team,runs,balls,strike_rate,dot_pct,boundary_pct,kkr_benchmark,kkr_benchmark_sr,sr_advantage_vs_benchmark,advantage,risk,classification
0,2023,Death,Vijay Shankar,Gujarat Titans,122,47,259.57,17.02,38.30,Andre Russell,163.41,96.16,Exceptional historical scoring rate,Only 47 balls; unusually high rate may not per...,Monitor
1,2023,Middle,Suryakumar Yadav,Mumbai Indians,450,246,182.93,25.20,28.05,Venkatesh Iyer,160.27,22.66,"Strong scoring rate, volume and dot-ball avoid...",Role compatibility and availability not establ...,Priority candidate
2,2023,Powerplay,Yashasvi Jaiswal,Rajasthan Royals,361,206,175.24,46.60,34.47,Jason Roy,146.28,28.96,Higher scoring rate with substantial workload,High dot-ball percentage; role and availabilit...,Priority candidate
3,2024,Death,Tristan Stubbs,Delhi Capitals,252,96,262.50,11.46,39.58,Andre Russell,210.94,51.56,"Exceptional scoring rate, boundary frequency a...",Limited sample; squad fit and availability unk...,Priority candidate
4,2024,Middle,Rajat Patidar,Royal Challengers Bengaluru,312,172,181.40,28.49,22.67,Shreyas Iyer,140.13,41.27,Strong scoring rate with a meaningful workload,Strike rate alone does not establish role equi...,Priority candidate
5,2024,Powerplay,Travis Head,Sunrisers Hyderabad,402,193,208.29,33.16,38.34,Phil Salt,185.00,23.29,High scoring rate supported by substantial volume,KKR already had a productive opening combination,Monitor


BOWLING SCOUTING DECISIONS


,year,phase,candidate,team,legal_balls,runs_conceded,wickets,economy,dot_pct,boundary_balls_per_100,wides,no_ball_extras,kkr_benchmark,kkr_benchmark_economy,economy_difference_vs_benchmark,advantage,risk,classification
0,2023,Middle,Harpreet Brar,Punjab Kings,90,93,7,6.20,37.78,10.00,1,0,Sunil Narine,7.42,-1.22,Strong containment and low boundary frequency,Only 15 overs; wicket-taking sustainability un...,Monitor
1,2023,Powerplay,Trent Boult,Rajasthan Royals,156,177,10,6.81,55.77,19.23,6,0,Harshit Rana,7.93,-1.12,Excellent economy and dot-ball percentage,Smaller workload than Shami; availability unknown,Priority candidate
2,2023,Powerplay,Mohammed Shami,Gujarat Titans,264,330,17,7.50,55.68,19.70,16,1,Harshit Rana,7.93,-0.43,"Strong wicket output, economy and dot-ball pre...",16 wides and one no-ball; historical season only,Priority candidate
3,2024,Death,Jasprit Bumrah,Mumbai Indians,101,109,11,6.48,34.65,9.90,2,0,Andre Russell,9.64,-3.16,Outstanding run prevention and boundary suppre...,Limited phase workload; availability and acqui...,Priority candidate
4,2024,Death,Harshal Patel,Punjab Kings,126,218,16,10.38,28.57,21.43,13,0,Andre Russell,9.64,0.74,High wicket output,"Expensive overs, frequent boundaries and 13 wides",Monitor
5,2024,Middle,Matheesha Pathirana,Chennai Super Kings,66,71,8,6.45,45.45,9.09,12,0,Varun Chakravarthy,7.68,-1.23,High wicket output and excellent boundary supp...,Only 11 overs and 12 wides; limited evidence,Monitor


In [49]:
print("BATTING CLASSIFICATION COUNTS")
display(
    batting_decision_df["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="candidate_count")
)

print("BOWLING CLASSIFICATION COUNTS")
display(
    bowling_decision_df["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="candidate_count")
)

print("PRIORITY BATTING PROFILES")
display(
    batting_decision_df.loc[
        batting_decision_df["classification"].eq("Priority candidate"),
        ["year", "phase", "candidate", "runs", "balls", "strike_rate", "advantage"]
    ].reset_index(drop=True)
)

print("PRIORITY BOWLING PROFILES")
display(
    bowling_decision_df.loc[
        bowling_decision_df["classification"].eq("Priority candidate"),
        [
            "year", "phase", "candidate", "legal_balls",
            "wickets", "economy", "dot_pct", "advantage"
        ]
    ].reset_index(drop=True)
)

BATTING CLASSIFICATION COUNTS


,classification,candidate_count
0,Priority candidate,4
1,Monitor,2


BOWLING CLASSIFICATION COUNTS


,classification,candidate_count
0,Priority candidate,3
1,Monitor,3


PRIORITY BATTING PROFILES


,year,phase,candidate,runs,balls,strike_rate,advantage
0,2023,Powerplay,Yashasvi Jaiswal,361,206,175.24,Higher scoring rate with substantial workload
1,2023,Middle,Suryakumar Yadav,450,246,182.93,"Strong scoring rate, volume and dot-ball avoid..."
2,2024,Middle,Rajat Patidar,312,172,181.40,Strong scoring rate with a meaningful workload
3,2024,Death,Tristan Stubbs,252,96,262.50,"Exceptional scoring rate, boundary frequency a..."


PRIORITY BOWLING PROFILES


,year,phase,candidate,legal_balls,wickets,economy,dot_pct,advantage
0,2023,Powerplay,Mohammed Shami,264,17,7.50,55.68,"Strong wicket output, economy and dot-ball pre..."
1,2023,Powerplay,Trent Boult,156,10,6.81,55.77,Excellent economy and dot-ball percentage
2,2024,Death,Jasprit Bumrah,101,11,6.48,34.65,Outstanding run prevention and boundary suppre...
